# Resolución de Actividad: Optimización, Validación y Sesgo

Este notebook contiene el código fuente para cumplir con los requerimientos de la actividad.

> **Nota Importante:** Como no se proporcionó el código o dataset del deber anterior, este notebook utiliza un dataset de ejemplo (`Breast Cancer` de scikit-learn) y un modelo `RandomForestClassifier` para que todo el código sea funcional y ejecutable desde el primer momento. 
>
> **Debes reemplazar la sección de carga de datos** con tu propio dataset y tu modelo inicial.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.inspection import PartialDependenceDisplay
from sklearn.datasets import load_breast_cancer
import warnings
warnings.filterwarnings('ignore')

## Fase 1: Identificación de problemas de generalización
Evaluamos el modelo base para detectar sobreajuste o subajuste comparando el rendimiento en entrenamiento vs validación.

In [ ]:
# 1. CARGA DE DATOS Y PREPARACIÓN BASE
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entrenar modelo base (con hiperparámetros por defecto)
base_model = RandomForestClassifier(random_state=42)
base_model.fit(X_train, y_train)

# Evaluar rendimiento para detectar generalización
train_preds = base_model.predict(X_train)
test_preds = base_model.predict(X_test)

print(f"Precisión en Entrenamiento (Accuracy): {accuracy_score(y_train, train_preds):.4f}")
print(f"Precisión en Prueba (Accuracy): {accuracy_score(y_test, test_preds):.4f}")
print("\nAnálisis: Si la precisión en entrenamiento es muy cercana a 1.0 pero en prueba es significativamente menor, indica sobreajuste (Overfitting).")

## Fase 2: Estrategias de optimización de hiperparámetros
Utilizamos `GridSearchCV` para buscar sistemáticamente la mejor combinación de parámetros.

In [ ]:
# Definir el espacio de búsqueda de hiperparámetros críticos
param_grid = {
    'n_estimators': [50, 100, 200],  # Número de árboles
    'max_depth': [None, 10, 20],     # Profundidad máxima del árbol
    'min_samples_split': [2, 5]      # Muestras mínimas para dividir un nodo
}

print("Iniciando optimización con GridSearchCV...")
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_

print("\nMejores hiperparámetros encontrados:", grid_search.best_params_)
print(f"Accuracy con modelo optimizado (Prueba): {accuracy_score(y_test, best_model.predict(X_test)):.4f}")

## Fase 3: Análisis y Componentes
### 3.1 Importancia de características (Variables que más influyen)
Esto nos ayuda a entender en qué se enfoca el modelo.

In [ ]:
importances = best_model.feature_importances_
indices = np.argsort(importances)[::-1][:10] # Tomamos el Top 10

plt.figure(figsize=(10, 5))
plt.title("Top 10 Características más importantes")
plt.bar(range(10), importances[indices], align="center")
plt.xticks(range(10), X.columns[indices], rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 3.2 Partial Dependence Plots (PDP) y Efectos de Interacción
Analizamos el efecto individual de las principales variables y cómo se combinan (interacción 2D).

In [ ]:
print("Generando Partial Dependence Plots...")
# Usaremos las 2 características más importantes para el análisis
features_to_plot = [X.columns[indices[0]], X.columns[indices[1]]]

fig, ax = plt.subplots(figsize=(12, 5))
PartialDependenceDisplay.from_estimator(best_model, X_train, features_to_plot, ax=ax)
plt.suptitle('Partial Dependence Plots (Efectos individuales)')
plt.show()

print("Generando gráfico de interacción (2D PDP)...")
fig, ax = plt.subplots(figsize=(8, 6))
interaction_features = [(X.columns[indices[0]], X.columns[indices[1]])]
PartialDependenceDisplay.from_estimator(best_model, X_train, interaction_features, ax=ax)
plt.suptitle('Efecto de interacción entre las dos características principales')
plt.show()

## Fase 4: Detección y mitigación de sesgo
Para probar el sesgo, simularemos una variable sensible (ej. Grupos demográficos A y B) en nuestro conjunto de datos.

In [ ]:
np.random.seed(42)
X_test_with_sensitive = X_test.copy()
X_test_with_sensitive['grupo_sensible'] = np.random.choice(['Grupo A', 'Grupo B'], size=len(X_test))

preds = best_model.predict(X_test)

# Evaluar rendimiento segmentado por el grupo
grupo_A_mask = X_test_with_sensitive['grupo_sensible'] == 'Grupo A'
grupo_B_mask = X_test_with_sensitive['grupo_sensible'] == 'Grupo B'

acc_A = accuracy_score(y_test[grupo_A_mask], preds[grupo_A_mask])
acc_B = accuracy_score(y_test[grupo_B_mask], preds[grupo_B_mask])

print(f"Rendimiento en Grupo A (Accuracy): {acc_A:.4f}")
print(f"Rendimiento en Grupo B (Accuracy): {acc_B:.4f}")

diferencia = abs(acc_A - acc_B)
print(f"\nDiferencia de precisión entre grupos: {diferencia:.4f}")
if diferencia > 0.10:
    print("Alerta: Se detecta un posible sesgo significativo.")
else:
    print("El modelo parece ser equitativo en este contexto simulado.")

## Fase 5: Validación en contextos simulados (Robustez)
Realizamos una prueba de estrés (Stress Testing) introduciendo ruido a las variables de entrada para ver si el modelo mantiene su capacidad predictiva en un entorno menos ideal.

In [ ]:
print("Realizando prueba de estrés añadiendo ruido gaussiano a los datos...")
# Añadir ruido proporcional a la desviación estándar de cada variable
ruido = np.random.normal(0, X_test.std() * 0.5, X_test.shape)
X_test_ruidoso = X_test + ruido

preds_ruidosas = best_model.predict(X_test_ruidoso)
acc_ruido = accuracy_score(y_test, preds_ruidosas)

print(f"Accuracy original (Prueba limpia): {accuracy_score(y_test, preds):.4f}")
print(f"Accuracy con ruido (Prueba de Estrés): {acc_ruido:.4f}")

caida = accuracy_score(y_test, preds) - acc_ruido
print(f"\nCaída en precisión: {caida:.4f}")
print("Si la caída es drástica, el modelo podría no ser robusto para operar en la vida real con datos ruidosos o anómalos.")